In [ ]:
!pip install pdfplumber spacy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 50.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving resume-sample(5).pdf to resume-sample(5).pdf
Saving resume-sample(4).pdf to resume-sample(4).pdf
Saving resume-sample(3).pdf to resume-sample(3).pdf
Saving resume-sample(2).pdf to resume-sample(2).pdf
Saving resume-sample(1).pdf to resume-sample(1).pdf


In [ ]:
!pip install pdfplumber sentence-transformers scikit-learn

In [ ]:
import pdfplumber

def extract_text_from_pdf(file_name):
    text = ""
    with pdfplumber.open(file_name) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
job_description = """
Looking for a professional with experience in management, communication,
data handling, reporting, and organizational skills.
"""

In [ ]:
def extract_skills(text):
    skills_list = [
        "python","java","c++","sql","excel","accounting",
        "finance","quickbooks","communication","management"
    ]

    text = text.lower()
    return [skill for skill in skills_list if skill in text]

In [ ]:
def skill_score(resume_skills, jd):
    jd = jd.lower()
    match = sum(1 for skill in resume_skills if skill in jd)
    return match / len(resume_skills) if resume_skills else 0

In [ ]:
def experience_score(text):
    import re

    matches = re.findall(r'(\d+)\s*(?:\+)?\s*years', text.lower())

    if matches:
        years = max([int(x) for x in matches])
        return min(years / 10, 1)

    # 🔥 fallback using keywords
    if "intern" in text.lower():
        return 0.3
    elif "project" in text.lower():
        return 0.2

    return 0.1

In [ ]:
def generate_justification(candidate):
    skills = candidate['skills']
    semantic = candidate['semantic']
    skill_score_val = candidate['skill_score']
    exp = candidate['experience_score']

    if candidate['score'] > 50:
        return f"Strong candidate with good semantic alignment ({semantic}%) and relevant skills {skills[:3]}."

    elif skill_score_val > 50:
        return f"Candidate has relevant skills {skills[:2]} but moderate semantic alignment ({semantic}%)."

    elif semantic > 45:
        return f"Good contextual understanding but lacks key required skills."

    else:
        return f"Low match due to weak alignment in both skills and experience."

In [ ]:
import re
from sklearn.metrics.pairwise import cosine_similarity

# 🔹 Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

# 🔹 Split into sentences
def split_sentences(text):
    sentences = re.split(r'\.', text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

results = []

# 🔹 Encode JD
jd_clean = clean_text(job_description)
jd_embedding = model.encode(jd_clean)

for file_name in uploaded.keys():

    resume_text = extract_text_from_pdf(file_name)

    # 🔹 Clean resume
    resume_clean = clean_text(resume_text)

    # 🔹 Sentence-level comparison
    sentences = split_sentences(resume_clean)

    similarities = []

    for sentence in sentences:
        sent_embedding = model.encode(sentence)
        sim = cosine_similarity([sent_embedding], [jd_embedding])[0][0]
        similarities.append(sim)

    # 🔥 Semantic score (FIXED)
    semantic = max(similarities) if similarities else 0

    # 🔹 Skill + Experience scores
    skills = extract_skills(resume_text)
    skill = skill_score(skills, job_description)
    experience = experience_score(resume_text)

    # 🔥 FINAL HYBRID SCORE
    final_score_combined = (
        0.5 * semantic +
        0.3 * skill +
        0.2 * experience
    )

    results.append({
        "name": file_name,
        "score": round(final_score_combined * 100, 2),
        "skills": skills,
        "semantic": round(semantic * 100, 2),
        "skill_score": round(skill * 100, 2),
        "experience_score": round(experience * 100, 2)
    })

# 🔥 Sort results
ranked_results = sorted(results, key=lambda x: x["score"], reverse=True)

# 🔥 Display
print("\n🏆 FINAL HYBRID RANKING:\n")

for i, candidate in enumerate(ranked_results, start=1):
    print(f"{i}. {candidate['name']}")
    print(f"   Final Score: {candidate['score']:.2f}%")
    print(f"   Semantic: {candidate['semantic']:.2f}%")
    print(f"   Skill: {candidate['skill_score']:.2f}%")
    print(f"   Experience: {candidate['experience_score']:.2f}%")
    print(f"   Skills: {candidate['skills']}")
    print(f"   Justification: {generate_justification(candidate)}")
    print("-" * 40)


🏆 FINAL HYBRID RANKING:

1. resume-sample(5).pdf
   Final Score: 46.74%
   Semantic: 49.48%
   Skill: 66.67%
   Experience: 10.00%
   Skills: ['excel', 'communication', 'management']
   Justification: Candidate has relevant skills ['excel', 'communication'] but moderate semantic alignment (49.47999954223633%).
----------------------------------------
2. resume-sample(3).pdf
   Final Score: 45.21%
   Semantic: 46.41%
   Skill: 66.67%
   Experience: 10.00%
   Skills: ['excel', 'communication', 'management']
   Justification: Candidate has relevant skills ['excel', 'communication'] but moderate semantic alignment (46.40999984741211%).
----------------------------------------
3. resume-sample(2).pdf
   Final Score: 38.99%
   Semantic: 35.98%
   Skill: 50.00%
   Experience: 30.00%
   Skills: ['excel', 'communication']
   Justification: Low match due to weak alignment in both skills and experience.
----------------------------------------
4. resume-sample(4).pdf
   Final Score: 38.18%
   Se